# VAE vs VQ-VAE Watermarking Performance Comparison (Improved)

## 실험 목표
1. **semantic_wm 데이터셋**으로 VAE와 VQ-VAE 학습
2. **Latent 차원**: 100D 고정
3. **Beta 값 비교 실험**: β = 0.01, 0.1, 0.5, 1.0
4. **개선사항**:
   - ✅ Decoder L2 정규화 적용
   - ✅ BatchNorm1d 사용
   - ✅ Dropout = 0.2
5. **평가 지표**: Watermark → Reconstructed CLIP Cosine Distance

## 비교 모델
```
1. VAE-β=0.01:  Semantic preservation (거의 AE)
2. VAE-β=0.1:   Weak regularization
3. VAE-β=0.5:   Moderate regularization
4. VAE-β=1.0:   Standard VAE (표준정규분포 강제)
5. VQ-VAE:      Discrete latent (512 codebook)
```

## 파이프라인
```
Training:
  semantic_wm images → CLIP(512D) → VAE/VQ-VAE → Latent(100D)
  
Testing:
  Test image → CLIP(512D) → Latent(100D) → Watermark bits
            ↓
  Decoder → Reconstructed CLIP(512D) [L2 normalized]
            ↓
  Evaluation: Cosine Distance = 1 - Cosine Similarity
```

## 1. 환경 설정 및 패키지 설치

In [ ]:
# Core ML libraries
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# Computer Vision & Image Processing
!pip install -q numpy>=2.0 scipy>=1.13 pillow==10.4.0 opencv-python
!pip install -q scikit-image scikit-learn matplotlib seaborn

In [ ]:
# Deep Learning utilities
!pip install -q einops==0.8.0 timm==0.9.12
!pip install -q tqdm easydict

In [ ]:
# Transformers and CLIP
!pip install -q transformers==4.45.2 open-clip-torch==2.26.1

In [ ]:
print("✅ All packages installed successfully!")

In [ ]:
# 런타임 재시작
import os
os.kill(os.getpid(), 9)

## 2. 라이브러리 Import

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

# GPU 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

## 3. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted")

In [ ]:
# semantic_wm 데이터셋 압축 해제
zip_path = '/content/drive/MyDrive/semantic_wm/dataset.zip'

print("압축 해제 중...")
!unzip -q {zip_path} -d /content/semantic_wm/
print("✅ 데이터셋 압축 해제 완료")

# 데이터셋 구조 확인
print("\n=== 데이터셋 구조 ===")
!ls -R /content/semantic_wm/dataset/train/

## 4. 데이터셋 경로 설정

In [ ]:
# 데이터셋 경로
train_path = '/content/semantic_wm/dataset/train'
test_path = '/content/semantic_wm/dataset/test'

# 카테고리 및 최대 이미지 수
categories = ['normal', 'violence', 'sexual']
max_images_per_category = None  # None = 전체 사용

print("✅ 데이터셋 경로 설정 완료")
print(f"   - Train: {train_path}")
print(f"   - Test: {test_path}")
print(f"   - Categories: {categories}")

In [ ]:
# 데이터셋 경로 확인 (디버깅)
import os

print("\n" + "="*80)
print("데이터셋 경로 확인")
print("="*80)

# Train 경로 확인
if os.path.exists(train_path):
    print(f"✅ Train 경로 존재: {train_path}")
    train_subdirs = os.listdir(train_path)
    print(f"   하위 폴더/파일: {train_subdirs}")

    for category in categories:
        cat_path = os.path.join(train_path, category)
        if os.path.exists(cat_path):
            files = [f for f in os.listdir(cat_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))]
            print(f"   [{category}] 이미지 파일 수: {len(files)}")
            if len(files) > 0:
                print(f"      예시: {files[:3]}")
        else:
            print(f"   ⚠️ [{category}] 경로 없음: {cat_path}")
else:
    print(f"❌ Train 경로가 존재하지 않습니다: {train_path}")
    print("\n가능한 대안 경로:")
    possible_paths = [
        './semantic_wm/dataset/train',
        '../semantic_wm/dataset/train',
        './dataset/train'
    ]
    for p in possible_paths:
        if os.path.exists(p):
            print(f"   ✅ {p}")

print("\n" + "-"*80)

# Test 경로 확인
if os.path.exists(test_path):
    print(f"✅ Test 경로 존재: {test_path}")
    test_subdirs = os.listdir(test_path)
    print(f"   하위 폴더/파일: {test_subdirs}")

    for category in categories:
        cat_path = os.path.join(test_path, category)
        if os.path.exists(cat_path):
            files = [f for f in os.listdir(cat_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))]
            print(f"   [{category}] 이미지 파일 수: {len(files)}")
            if len(files) > 0:
                print(f"      예시: {files[:3]}")
        else:
            print(f"   ⚠️ [{category}] 경로 없음: {cat_path}")
else:
    print(f"❌ Test 경로가 존재하지 않습니다: {test_path}")

print("="*80)

## 5. CLIP 모델 로드

In [ ]:
# CLIP 모델 로드
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Evaluation mode
clip_model.eval()

print("✅ CLIP 모델 로드 완료")
print(f"   - Model: openai/clip-vit-base-patch32")
print(f"   - Device: {device}")

## 6. 데이터 로드 함수

In [ ]:
def load_images_from_category(base_path, category, max_images=None):
    """카테고리별 이미지 경로 로드"""
    category_path = os.path.join(base_path, category)

    if not os.path.exists(category_path):
        print(f"⚠️ 카테고리 경로가 존재하지 않습니다: {category_path}")
        return []

    image_paths = []

    for img_name in os.listdir(category_path):
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
            full_path = os.path.join(category_path, img_name)
            # 파일이 실제로 존재하고 크기가 0이 아닌지 확인
            if os.path.isfile(full_path) and os.path.getsize(full_path) > 0:
                image_paths.append(full_path)

    if max_images:
        image_paths = image_paths[:max_images]

    print(f"   [{category}] {len(image_paths)}개 이미지 발견")
    return image_paths


def extract_clip_embeddings(image_paths, model, processor, device, batch_size=32):
    """이미지 경로들로부터 CLIP embedding 추출"""
    embeddings = []

    for i in tqdm(range(0, len(image_paths), batch_size), desc="Extracting embeddings"):
        batch_paths = image_paths[i:i+batch_size]

        # 이미지 로드 시 에러 처리
        images = []
        valid_paths = []
        for p in batch_paths:
            try:
                img = Image.open(p).convert('RGB')
                images.append(img)
                valid_paths.append(p)
            except Exception as e:
                print(f"\n⚠️ 이미지 로드 실패: {p}")
                print(f"   Error: {str(e)}")
                continue

        if len(images) == 0:
            continue

        inputs = processor(images=images, return_tensors="pt", padding=True).to(device)

        with torch.no_grad():
            image_features = model.get_image_features(**inputs)
            # L2 정규화 (CLIP은 기본적으로 정규화된 embedding 생성)
            image_features = F.normalize(image_features, p=2, dim=1)

        embeddings.append(image_features.cpu().numpy())

    if len(embeddings) == 0:
        raise ValueError("유효한 이미지가 하나도 없습니다. 데이터셋 경로를 확인하세요.")

    return np.vstack(embeddings)


print("✅ 데이터 로드 함수 정의 완료")
print("   - load_images_from_category: 카테고리별 이미지 경로 로드")
print("   - extract_clip_embeddings: CLIP embedding 추출 (L2 정규화)")

## 7. Training 데이터 로드

In [ ]:
train_embeddings = []
train_labels = []
train_category_stats = {}

print("\n" + "="*80)
print("Training Data 로드 중...")
print("="*80)

for category in categories:
    print(f"\n[{category.upper()}] 카테고리 처리 중...")

    # 이미지 경로 로드
    image_paths = load_images_from_category(train_path, category, max_images_per_category)
    print(f"  - 로드된 이미지 수: {len(image_paths)}")

    # CLIP 임베딩 추출
    embeddings = extract_clip_embeddings(image_paths, clip_model, clip_processor, device)

    train_embeddings.append(embeddings)
    train_labels.extend([category] * len(embeddings))
    train_category_stats[category] = len(embeddings)

    print(f"  - 추출된 임베딩 shape: {embeddings.shape}")
    print(f"  ✓ {category} 완료")

# 전체 데이터 결합
train_embeddings = np.vstack(train_embeddings)
train_labels = np.array(train_labels)

print("\n" + "="*80)
print("Training Data 로드 완료")
print("="*80)
print(f"전체 임베딩 shape: {train_embeddings.shape}")
print(f"전체 레이블 수: {len(train_labels)}")
print(f"평균 L2 norm: {np.linalg.norm(train_embeddings, axis=1).mean():.6f}")
print("\n카테고리별 통계:")
for cat, count in train_category_stats.items():
    print(f"  - {cat.capitalize()}: {count} images ({count/len(train_labels)*100:.1f}%)")
print("="*80)

## 8. Test 데이터 로드

In [ ]:
test_embeddings = []
test_labels = []
test_category_stats = {}

print("\n" + "="*80)
print("Test Data 로드 중...")
print("="*80)

for category in categories:
    print(f"\n[{category.upper()}] 카테고리 처리 중...")

    # 이미지 경로 로드
    image_paths = load_images_from_category(test_path, category, max_images_per_category)
    print(f"  - 로드된 이미지 수: {len(image_paths)}")

    # CLIP 임베딩 추출
    embeddings = extract_clip_embeddings(image_paths, clip_model, clip_processor, device)

    test_embeddings.append(embeddings)
    test_labels.extend([category] * len(embeddings))
    test_category_stats[category] = len(embeddings)

    print(f"  - 추출된 임베딩 shape: {embeddings.shape}")
    print(f"  ✓ {category} 완료")

# 전체 데이터 결합
test_embeddings = np.vstack(test_embeddings)
test_labels = np.array(test_labels)

print("\n" + "="*80)
print("Test Data 로드 완료")
print("="*80)
print(f"전체 임베딩 shape: {test_embeddings.shape}")
print(f"전체 레이블 수: {len(test_labels)}")
print(f"평균 L2 norm: {np.linalg.norm(test_embeddings, axis=1).mean():.6f}")
print("\n카테고리별 통계:")
for cat, count in test_category_stats.items():
    print(f"  - {cat.capitalize()}: {count} images ({count/len(test_labels)*100:.1f}%)")
print("="*80)

## 9. VAE 모델 정의 (개선 버전)

In [ ]:
class ImprovedVAE(nn.Module):
    """
    Improved VAE for CLIP Embeddings

    개선사항:
    - BatchNorm1d 사용 (LayerNorm 대신)
    - Decoder 출력에 L2 정규화 적용
    - Dropout 0.2 (더 강한 정규화)
    """
    def __init__(self, input_dim=512, latent_dim=100):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim

        # Encoder: CLIP(512) → Latent(latent_dim)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

        # Decoder: Latent(latent_dim) → CLIP'(512)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        recon = self.decoder(z)
        # ✅ L2 정규화 적용 (중요!)
        return F.normalize(recon, p=2, dim=1)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar, z

    def get_latent(self, x):
        """Inference용: latent 추출"""
        mu, _ = self.encode(x)
        return mu


print("✅ Improved VAE 모델 정의 완료")
print("   - Input: 512D (CLIP)")
print("   - Latent: 100D")
print("   - Output: 512D (L2 normalized)")
print("   - BatchNorm1d + Dropout 0.2")

## 10. VQ-VAE 모델 정의 (개선 버전)

In [ ]:
class VectorQuantizer(nn.Module):
    """Vector Quantization Layer"""
    def __init__(self, num_embeddings, embedding_dim, commitment_cost=0.25):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost

        # Codebook
        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        self.embeddings.weight.data.uniform_(-1/num_embeddings, 1/num_embeddings)

    def forward(self, z):
        # Calculate distances to all codebook vectors
        distances = torch.cdist(z, self.embeddings.weight)

        # Find nearest codebook vector
        encoding_indices = torch.argmin(distances, dim=1)

        # Get quantized vectors
        quantized = self.embeddings(encoding_indices)

        # VQ Loss
        codebook_loss = F.mse_loss(quantized, z.detach())
        commitment_loss = F.mse_loss(z, quantized.detach())
        vq_loss = codebook_loss + self.commitment_cost * commitment_loss

        # Straight-through estimator
        quantized = z + (quantized - z).detach()

        return quantized, vq_loss, encoding_indices


class ImprovedVQVAE(nn.Module):
    """
    Improved VQ-VAE for CLIP Embeddings

    개선사항:
    - BatchNorm1d 사용
    - Decoder 출력에 L2 정규화 적용
    - Dropout 0.2
    """
    def __init__(self, input_dim=512, latent_dim=100, num_embeddings=512):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.num_embeddings = num_embeddings

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )

        # Vector Quantizer
        self.vq = VectorQuantizer(num_embeddings, latent_dim)

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, input_dim)
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        recon = self.decoder(z)
        # ✅ L2 정규화 적용 (중요!)
        return F.normalize(recon, p=2, dim=1)

    def forward(self, x):
        z = self.encode(x)
        quantized, vq_loss, encoding_indices = self.vq(z)
        recon = self.decode(quantized)
        return recon, vq_loss, encoding_indices

    def get_latent(self, x):
        """Inference용: quantized latent 추출"""
        z = self.encode(x)
        quantized, _, encoding_indices = self.vq(z)
        return quantized, encoding_indices


print("✅ Improved VQ-VAE 모델 정의 완료")
print("   - Input: 512D (CLIP)")
print("   - Latent: 100D (Discrete)")
print("   - Codebook: 512 entries")
print("   - Output: 512D (L2 normalized)")
print("   - BatchNorm1d + Dropout 0.2")

## 11. Loss 함수 정의

In [ ]:
def vae_loss(recon, target, mu, logvar, beta=0.01):
    """
    VAE Loss = Reconstruction Loss + β × KL Divergence

    Args:
        recon: Reconstructed CLIP embedding (L2 normalized)
        target: Original CLIP embedding (L2 normalized)
        mu: Mean of latent distribution
        logvar: Log variance of latent distribution
        beta: Weight for KL divergence (작은 값으로 클러스터 유지)

    Returns:
        total_loss, recon_loss, kl_loss
    """
    # 1. Reconstruction loss (Cosine Distance) - CLIP에 최적화!
    cosine_sim = F.cosine_similarity(recon, target, dim=1).mean()
    recon_loss = 1 - cosine_sim  # Cosine Distance

    # 2. KL divergence
    # kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    # Total loss
    total_loss = recon_loss + beta * kl_loss

    return total_loss, recon_loss, kl_loss


def vqvae_loss(recon, target, vq_loss):
    """
    VQ-VAE Loss = Reconstruction Loss + VQ Loss

    Args:
        recon: Reconstructed CLIP embedding (L2 normalized)
        target: Original CLIP embedding (L2 normalized)
        vq_loss: Vector quantization loss

    Returns:
        total_loss, recon_loss
    """
    # Reconstruction loss (Cosine Distance) - CLIP에 최적화!
    cosine_sim = F.cosine_similarity(recon, target, dim=1).mean()
    recon_loss = 1 - cosine_sim  # Cosine Distance

    # Total loss
    total_loss = recon_loss + vq_loss

    return total_loss, recon_loss


print("✅ Loss 함수 정의 완료")
print("   - vae_loss: Cosine Distance + β×KLD (β=0.01)")
print("   - vqvae_loss: Cosine Distance + VQ")

## 12. 데이터셋 및 DataLoader 준비

In [ ]:
# 데이터셋 클래스
class CLIPEmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = torch.FloatTensor(embeddings)
        self.labels = labels

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


# Train/Val split
X_train, X_val, y_train, y_val = train_test_split(
    train_embeddings, train_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_labels
)

# Dataset 생성
train_dataset = CLIPEmbeddingDataset(X_train, y_train)
val_dataset = CLIPEmbeddingDataset(X_val, y_val)

# DataLoader 생성
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print("✅ 데이터셋 준비 완료")
print(f"   - 학습 데이터: {len(train_dataset)} samples")
print(f"   - 검증 데이터: {len(val_dataset)} samples")
print(f"   - Batch size: {batch_size}")
print(f"   - Train batches: {len(train_loader)}")
print(f"   - Val batches: {len(val_loader)}")

## 13. Training 함수

In [ ]:
def train_vae(model, train_loader, val_loader, epochs=50, lr=1e-3, beta=0.01):
    """VAE Training 함수"""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {
        'train_losses': [], 'train_recon': [], 'train_kld': [],
        'val_losses': [], 'val_recon': [], 'val_kld': [], 'val_cosine': []
    }

    best_val_loss = float('inf')

    # Epoch progress bar
    epoch_pbar = tqdm(range(epochs), desc="VAE Training")

    for epoch in epoch_pbar:
        # Training
        model.train()
        train_loss, train_recon, train_kld = 0, 0, 0

        for batch_x, _ in train_loader:
            batch_x = batch_x.to(device)

            # Forward
            recon, mu, logvar, _ = model(batch_x)
            loss, recon_loss, kl_loss = vae_loss(recon, batch_x, mu, logvar, beta)

            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_recon += recon_loss.item()
            train_kld += kl_loss.item()

        train_loss /= len(train_loader)
        train_recon /= len(train_loader)
        train_kld /= len(train_loader)

        # Validation
        model.eval()
        val_loss, val_recon, val_kld = 0, 0, 0
        val_cosine_sims = []

        with torch.no_grad():
            for batch_x, _ in val_loader:
                batch_x = batch_x.to(device)

                recon, mu, logvar, _ = model(batch_x)
                loss, recon_loss, kl_loss = vae_loss(recon, batch_x, mu, logvar, beta)

                val_loss += loss.item()
                val_recon += recon_loss.item()
                val_kld += kl_loss.item()

                # Cosine similarity
                cos_sim = F.cosine_similarity(batch_x, recon, dim=1).mean()
                val_cosine_sims.append(cos_sim.item())

        val_loss /= len(val_loader)
        val_recon /= len(val_loader)
        val_kld /= len(val_loader)
        val_cosine = np.mean(val_cosine_sims)

        # Save history
        history['train_losses'].append(train_loss)
        history['train_recon'].append(train_recon)
        history['train_kld'].append(train_kld)
        history['val_losses'].append(val_loss)
        history['val_recon'].append(val_recon)
        history['val_kld'].append(val_kld)
        history['val_cosine'].append(val_cosine)

        # Update epoch progress bar
        epoch_pbar.set_postfix({
            'train_loss': f"{train_loss:.4f}",
            'val_loss': f"{val_loss:.4f}",
            'cosine': f"{val_cosine:.4f}"
        })

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), '/content/best_vae_model.pth')

    return history


def train_vqvae(model, train_loader, val_loader, epochs=50, lr=1e-3):
    """VQ-VAE Training 함수"""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {
        'train_losses': [], 'train_recon': [], 'train_vq': [],
        'val_losses': [], 'val_recon': [], 'val_vq': [], 'val_cosine': []
    }

    best_val_loss = float('inf')

    # Epoch progress bar
    epoch_pbar = tqdm(range(epochs), desc="VQ-VAE Training")

    for epoch in epoch_pbar:
        # Training
        model.train()
        train_loss, train_recon, train_vq = 0, 0, 0

        for batch_x, _ in train_loader:
            batch_x = batch_x.to(device)

            # Forward
            recon, vq_loss_val, _ = model(batch_x)
            loss, recon_loss = vqvae_loss(recon, batch_x, vq_loss_val)

            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_recon += recon_loss.item()
            train_vq += vq_loss_val.item()

        train_loss /= len(train_loader)
        train_recon /= len(train_loader)
        train_vq /= len(train_loader)

        # Validation
        model.eval()
        val_loss, val_recon, val_vq = 0, 0, 0
        val_cosine_sims = []

        with torch.no_grad():
            for batch_x, _ in val_loader:
                batch_x = batch_x.to(device)

                recon, vq_loss_val, _ = model(batch_x)
                loss, recon_loss = vqvae_loss(recon, batch_x, vq_loss_val)

                val_loss += loss.item()
                val_recon += recon_loss.item()
                val_vq += vq_loss_val.item()

                # Cosine similarity
                cos_sim = F.cosine_similarity(batch_x, recon, dim=1).mean()
                val_cosine_sims.append(cos_sim.item())

        val_loss /= len(val_loader)
        val_recon /= len(val_loader)
        val_vq /= len(val_loader)
        val_cosine = np.mean(val_cosine_sims)

        # Save history
        history['train_losses'].append(train_loss)
        history['train_recon'].append(train_recon)
        history['train_vq'].append(train_vq)
        history['val_losses'].append(val_loss)
        history['val_recon'].append(val_recon)
        history['val_vq'].append(val_vq)
        history['val_cosine'].append(val_cosine)

        # Update epoch progress bar
        epoch_pbar.set_postfix({
            'train_loss': f"{train_loss:.4f}",
            'val_loss': f"{val_loss:.4f}",
            'cosine': f"{val_cosine:.4f}"
        })

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), '/content/best_vqvae_model.pth')

    return history


print("✅ Training 함수 정의 완료")
print("   - train_vae: VAE 학습")
print("   - train_vqvae: VQ-VAE 학습")

## 14. VAE 모델 학습

In [ ]:
# VAE 모델 초기화
vae_model = ImprovedVAE(input_dim=512, latent_dim=100).to(device)

print("="*80)
print("VAE 모델 학습 시작")
print("="*80)
print(f"Model: ImprovedVAE (512D → 100D)")
print(f"Beta: 0.01 (작은 값으로 클러스터 유지)")
print(f"Epochs: 50")
print(f"Learning rate: 1e-3")
print("="*80)

# 학습 시작
vae_history = train_vae(
    model=vae_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=50,
    lr=1e-3,
    beta=0.01
)

print("\n" + "="*80)
print("VAE 학습 완료!")
print("="*80)
print(f"Best Val Loss: {min(vae_history['val_losses']):.4f}")
print(f"Best Val Cosine: {max(vae_history['val_cosine']):.4f}")
print("="*80)

## 14-1. Beta 값 비교 실험 (β = 0.00, 0.001, 0.1)

In [ ]:
# Beta 값 비교 실험
beta_values = [0.0, 0.001, 0.1]
beta_models = {}
beta_histories = {}

print("="*80)
print("Beta 값 비교 실험 시작")
print("="*80)
print(f"실험 Beta 값: {beta_values}")
print(f"비교 대상: β=0.01 (이미 학습 완료)")
print("="*80)

for beta in beta_values:
    print(f"\n{'='*80}")
    print(f"Beta = {beta} VAE 학습 시작")
    print(f"{'='*80}")

    # 모델 초기화
    model = ImprovedVAE(input_dim=512, latent_dim=100).to(device)

    # 학습
    history = train_vae(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=50,
        lr=1e-3,
        beta=beta
    )

    # 저장
    beta_models[beta] = model
    beta_histories[beta] = history

    print(f"\n✅ Beta={beta} 학습 완료")
    print(f"   Best Val Loss: {min(history['val_losses']):.4f}")
    print(f"   Best Val Cosine: {max(history['val_cosine']):.4f}")

print("\n" + "="*80)
print("모든 Beta 값 실험 완료!")
print("="*80)

# β=0.01 결과도 추가
beta_models[0.01] = vae_model
beta_histories[0.01] = vae_history

## 14-2. Beta 값별 Loss 비교 시각화

In [ ]:
# Beta 값별 Loss 비교
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# beta_colors = {0.001: '#3498db', 0.01: '#2ecc71', 0.1: '#e67e22', 1.0: '#e74c3c'}
# all_betas = [0.001, 0.01, 0.1, 1.0]
beta_colors = {0.0: '#3498db', 0.001: '#2ecc71', 0.01: '#e67e22', 0.1: '#e74c3c'}
all_betas = [0.0, 0.001, 0.01, 0.1]

# 1. Total Loss
ax = axes[0, 0]
for beta in all_betas:
    history = beta_histories[beta]
    ax.plot(history['val_losses'], label=f'β={beta}',
            linewidth=2, color=beta_colors[beta])
ax.set_title('Validation Total Loss Comparison', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 2. Reconstruction Loss
ax = axes[0, 1]
for beta in all_betas:
    history = beta_histories[beta]
    ax.plot(history['val_recon'], label=f'β={beta}',
            linewidth=2, color=beta_colors[beta])
ax.set_title('Validation Reconstruction Loss', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 3. KLD Loss
ax = axes[1, 0]
for beta in all_betas:
    history = beta_histories[beta]
    ax.plot(history['val_kld'], label=f'β={beta}',
            linewidth=2, color=beta_colors[beta])
ax.set_title('Validation KLD Loss', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('KLD', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')  # KLD는 차이가 크므로 log scale

# 4. Cosine Similarity
ax = axes[1, 1]
for beta in all_betas:
    history = beta_histories[beta]
    ax.plot(history['val_cosine'], label=f'β={beta}',
            linewidth=2, color=beta_colors[beta], marker='o', markersize=2)
ax.set_title('Validation Cosine Similarity', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Cosine Similarity', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Beta Value Comparison: Training Dynamics',
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('/content/beta_comparison_training.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Beta 비교 Loss 시각화 완료")

## 14-3. Beta 값별 Test 성능 비교

In [ ]:
# Watermark 변환 클래스 (Latent ↔ Watermark)
class LatentWatermarker:
    """
    Latent vector를 Watermark (bit array)로 변환하고 복원하는 클래스
    실제 이미지 삽입을 시뮬레이션하기 위해 quantization 수행
    """
    def __init__(self, latent_dim=100):
        """
        Args:
            latent_dim: Latent vector 차원 (100D)
            bits_per_value: 각 latent 값당 비트 수 (1 = sign bit만)
        """
        self.latent_dim = latent_dim

    def compute_latent_stats(self, latent_vectors):
      """
      latent vectors: shape(N, latent_dim)의 numpy array
      """
      return {
          'mean': latent_vectors.mean(axis=0),
          'std': latent_vectors.std(axis=0)
      }

    def latent_to_watermark(self, latent_vector):
        """
        latent_vector: shape (latent_dim,) numpy array
        Returns:
            watermark: 0/1 binary array, shape (latent_dim,)
        """
        watermark = (latent_vector > 0).astype(np.int32)

        return watermark

    def watermark_to_latent(self, watermark, latent_stats):
        """
        watermark: 0/1 array, shape (latent_dim,)
        latent_stats: {'mean': [...], 'std': [...]}

        Returns:
            restored_latent: shape (latent_dim,) numpy array
        """
        # 1) 0/1 → -1/+1 (sign 복원)
        restored_latent = watermark.astype(np.float32) * 2 - 1  # {0→-1, 1→+1}

        # 2) mean/std 기반 scale 복원 (semantic 보존에 가장 중요!)
        if latent_stats is not None:
            mean = latent_stats['mean']
            std = latent_stats['std']

            # 원래 latent의 geometry를 유지하는 핵심
            restored_latent = restored_latent * std + mean

        return restored_latent

# Watermarker 초기화
watermarker = LatentWatermarker(latent_dim=100)

print("="*80)
print("✅ Watermark 변환 클래스 정의 완료")
print("="*80)
print(f"   Latent dimension: {watermarker.latent_dim}")
print("="*80)

In [ ]:
# Test 데이터로 각 Beta 모델 평가 (Watermark 변환 포함)
test_embeddings_tensor = torch.FloatTensor(test_embeddings).to(device)

beta_test_results = {}

print("="*80)
print("Beta 값별 Test 데이터셋 평가 (Watermark Pipeline)")
print("="*80)
print("Pipeline: Original CLIP → Encoder → Latent → Watermark (100-bit) → Latent → Decoder → Reconstructed CLIP")
print("="*80)

for beta in all_betas:
    print(f"\n{'='*80}")
    print(f"[Beta={beta}] 평가 중...")
    print(f"{'='*80}")

    model = beta_models[beta]
    model.eval()

    # Step 1: 원본 CLIP → Latent 추출 (Encoder)
    print(f"  [1/5] Latent 추출 중...")
    with torch.no_grad():
        mu, logvar = model.encode(test_embeddings_tensor)
        latents = mu  # μ만 사용 (워터마크로 사용)

    latents_np = latents.cpu().numpy()
    print(f"        ✓ Latent shape: {latents_np.shape}")

    # Step 2: Latent 통계 계산 (복원에 필요)
    print(f"  [2/5] Latent 통계 계산 중...")
    latent_stats = watermarker.compute_latent_stats(latents_np)
    print(f"        ✓ Mean: {latent_stats['mean'][:5]} ...")
    print(f"        ✓ Std: {latent_stats['std'][:5]} ...")

    # Step 3: Latent → Watermark 변환 (100-bit)
    print(f"  [3/5] Watermark 생성 중...")
    watermarks = np.array([watermarker.latent_to_watermark(latent)
                           for latent in latents_np])
    print(f"        ✓ Watermark shape: {watermarks.shape}")
    print(f"        ✓ Sample watermark: {watermarks[0][:20]} ...")

    # Step 4: Watermark → Latent 복원 (정보 손실 발생)
    print(f"  [4/5] Latent 복원 중...")
    latents_restored = np.array([watermarker.watermark_to_latent(wm, latent_stats)
                                  for wm in watermarks])
    latents_restored_tensor = torch.FloatTensor(latents_restored).to(device)
    print(f"        ✓ Restored latent shape: {latents_restored.shape}")

    # Step 5: 복원된 Latent → CLIP 복원 (Decoder)
    print(f"  [5/5] CLIP 복원 중...")
    with torch.no_grad():
        recon = model.decode(latents_restored_tensor)

    recon_np = recon.cpu().numpy()
    print(f"        ✓ Reconstructed CLIP shape: {recon_np.shape}")

    # Cosine Similarity 계산
    cosine_sims = F.cosine_similarity(
        torch.FloatTensor(test_embeddings),
        torch.FloatTensor(recon_np),
        dim=1
    ).numpy()

    cosine_dist = 1 - cosine_sims

    # Latent 복원 정확도 (원본 vs 복원)
    latent_cosine = F.cosine_similarity(
        torch.FloatTensor(latents_np),
        torch.FloatTensor(latents_restored),
        dim=1
    ).numpy()

    # 카테고리별 통계
    category_stats = {}
    for category in categories:
        mask = test_labels == category
        cat_sims = cosine_sims[mask]
        category_stats[category] = {
            'mean': cat_sims.mean(),
            'std': cat_sims.std(),
            'count': np.sum(mask)
        }

    beta_test_results[beta] = {
        'latents': latents_np,
        'latents_restored': latents_restored,
        'watermarks': watermarks,
        'recon': recon_np,
        'cosine_sims': cosine_sims,
        'cosine_dist': cosine_dist,
        'latent_cosine': latent_cosine,
        'category_stats': category_stats
    }

    print(f"\n  ✅ Beta={beta} 완료")
    print(f"     ├─ CLIP Cosine Sim: {cosine_sims.mean():.6f} ± {cosine_sims.std():.6f}")
    print(f"     ├─ CLIP Cosine Dist: {cosine_dist.mean():.6f}")
    print(f"     └─ Latent 복원 정확도: {latent_cosine.mean():.6f}")

print("\n" + "="*80)
print("✅ 모든 Beta 값 평가 완료 (Watermark Pipeline)")
print("="*80)

## 14-4. Beta 값별 성능 요약 비교

In [ ]:
# 성능 요약 테이블 (Watermark Pipeline 포함)
print("\n" + "="*80)
print("Beta 값별 성능 요약 (Watermark Pipeline)")
print("="*80)

summary_data = []
for beta in all_betas:
    results = beta_test_results[beta]
    summary_data.append({
        'Beta': beta,
        'CLIP Cosine Sim': results['cosine_sims'].mean(),
        'CLIP Cosine Std': results['cosine_sims'].std(),
        'Latent Recovery': results['latent_cosine'].mean(),
        'Cosine Dist': results['cosine_dist'].mean(),
        'Final Val Loss': beta_histories[beta]['val_losses'][-1],
        'Final KLD': beta_histories[beta]['val_kld'][-1]
    })

summary_df = pd.DataFrame(summary_data)
print("\n전체 요약 (Watermark 변환 포함):")
print(summary_df.to_string(index=False))

# 카테고리별 성능 비교
print("\n" + "-"*80)
print("카테고리별 CLIP Cosine Similarity (Watermark Pipeline)")
print("-"*80)

for category in categories:
    print(f"\n[{category.capitalize()}]")
    for beta in all_betas:
        stats = beta_test_results[beta]['category_stats'][category]
        print(f"   β={beta:4.2f}: {stats['mean']:.6f} ± {stats['std']:.6f} (n={stats['count']})")

# 최고 성능 Beta 찾기
best_beta_cosine = max(all_betas, key=lambda b: beta_test_results[b]['cosine_sims'].mean())
best_beta_latent = max(all_betas, key=lambda b: beta_test_results[b]['latent_cosine'].mean())
best_beta_recon = min(all_betas, key=lambda b: beta_histories[b]['val_recon'][-1])

print("\n" + "="*80)
print("🏆 최고 성능 Beta 값 (Watermark Pipeline)")
print("="*80)
print(f"✅ CLIP Cosine Similarity 기준: β={best_beta_cosine}")
print(f"   → 평균 Cosine Sim: {beta_test_results[best_beta_cosine]['cosine_sims'].mean():.6f}")
print(f"\n✅ Latent 복원 정확도 기준: β={best_beta_latent}")
print(f"   → 평균 Latent Cosine: {beta_test_results[best_beta_latent]['latent_cosine'].mean():.6f}")
print(f"\n✅ Reconstruction Loss 기준: β={best_beta_recon}")
print(f"   → Final Val Recon Loss: {beta_histories[best_beta_recon]['val_recon'][-1]:.6f}")
print("\n💡 해석:")
print("   - Latent Recovery: Watermark→Latent 복원 정확도 (높을수록 정보 손실 적음)")
print("   - CLIP Cosine Sim: 최종 CLIP 복원 품질 (높을수록 semantic 보존 우수)")
print("="*80)

## 14-5. Beta 값별 Latent Space t-SNE 비교

In [ ]:
# Beta 값별 Latent Space t-SNE 비교
from sklearn.manifold import TSNE

print("="*80)
print("Beta 값별 Latent Space t-SNE 계산 중...")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
colors_map = {'normal': '#2ecc71', 'violence': '#e74c3c', 'sexual': '#9b59b6'}
markers_map = {'normal': 'o', 'violence': '^', 'sexual': 's'}

for idx, beta in enumerate(all_betas):
    print(f"\nβ={beta} t-SNE 계산 중...")

    latents = beta_test_results[beta]['latents']

    # t-SNE
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    latents_2d = tsne.fit_transform(latents)

    # Plot
    ax = axes[idx // 2, idx % 2]

    for category in categories:
        mask = test_labels == category
        ax.scatter(
            latents_2d[mask, 0], latents_2d[mask, 1],
            c=colors_map[category], marker=markers_map[category],
            label=f'{category.capitalize()} ({np.sum(mask)})',
            alpha=0.6, s=80, edgecolors='white', linewidth=0.5
        )

    # 성능 정보 추가
    mean_sim = beta_test_results[beta]['cosine_sims'].mean()
    final_kld = beta_histories[beta]['val_kld'][-1]

    ax.set_title(f'β={beta} Latent Space (Cosine Sim={mean_sim:.4f}, KLD={final_kld:.2f})',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('t-SNE Dimension 1', fontsize=10)
    ax.set_ylabel('t-SNE Dimension 2', fontsize=10)
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3, linestyle='--')

    print(f"✅ β={beta} 완료")

plt.suptitle('Beta Value Comparison: Latent Space Structure (Test Set)',
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('/content/beta_comparison_latent_tsne.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ 모든 t-SNE 시각화 완료")
print("="*80)

## 14-5-2. Beta 값별 Latent + Reconstructed 동시 비교 (t-SNE)

In [ ]:
# Beta 값별 Latent Space와 Reconstructed Embedding 동시 비교
print("="*80)
print("Beta 값별 Latent + Reconstructed t-SNE 계산 중...")
print("="*80)

# 각 Beta에 대해 2x3 그리드 (Original, Latent, Reconstructed)
for beta in all_betas:
    print(f"\n{'='*80}")
    print(f"β={beta} 처리 중...")
    print(f"{'='*80}")

    # 데이터 추출
    latents = beta_test_results[beta]['latents']
    recon = beta_test_results[beta]['recon']

    # t-SNE 계산
    print("  - Original CLIP t-SNE 계산...")
    tsne_original = TSNE(n_components=2, random_state=42, perplexity=30)
    original_2d = tsne_original.fit_transform(test_embeddings)

    print("  - Latent Space t-SNE 계산...")
    tsne_latent = TSNE(n_components=2, random_state=42, perplexity=30)
    latent_2d = tsne_latent.fit_transform(latents)

    print("  - Reconstructed CLIP t-SNE 계산...")
    tsne_recon = TSNE(n_components=2, random_state=42, perplexity=30)
    recon_2d = tsne_recon.fit_transform(recon)

    # 시각화
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    titles = [
        f'Original CLIP (512D)',
        f'Latent Space (100D) - β={beta}',
        f'Reconstructed CLIP (512D) - β={beta}'
    ]
    data_list = [original_2d, latent_2d, recon_2d]

    colors_map = {'normal': '#2ecc71', 'violence': '#e74c3c', 'sexual': '#9b59b6'}
    markers_map = {'normal': 'o', 'violence': '^', 'sexual': 's'}

    for ax, title, data_2d in zip(axes, titles, data_list):
        for category in categories:
            mask = test_labels == category
            ax.scatter(
                data_2d[mask, 0], data_2d[mask, 1],
                c=colors_map[category], marker=markers_map[category],
                label=f'{category.capitalize()} ({np.sum(mask)})',
                alpha=0.6, s=80, edgecolors='white', linewidth=0.5
            )
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('t-SNE Dimension 1', fontsize=11)
        ax.set_ylabel('t-SNE Dimension 2', fontsize=11)
        ax.legend(loc='best', fontsize=9)
        ax.grid(True, alpha=0.3, linestyle='--')

    # 성능 정보 추가
    mean_sim = beta_test_results[beta]['cosine_sims'].mean()
    final_kld = beta_histories[beta]['val_kld'][-1]

    plt.suptitle(f'β={beta}: Original → Latent → Reconstructed (Cosine Sim={mean_sim:.4f}, KLD={final_kld:.2f})',
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'/content/beta_{beta}_pipeline_tsne.png', dpi=300, bbox_inches='tight')
    plt.show()

    print(f"  ✅ β={beta} 완료")

print("\n" + "="*80)
print("✅ 모든 Pipeline t-SNE 시각화 완료")
print("="*80)

## 14-5-3. Beta 값 전체 비교 (2x4 Grid: Latent vs Reconstructed)

In [ ]:
# 모든 Beta 값 한눈에 비교: Latent Space vs Reconstructed
print("="*80)
print("Beta 값 전체 비교 t-SNE (Latent + Reconstructed)")
print("="*80)

# Latent Space 비교 (2x2)
print("\n[1/2] Latent Space 비교 계산 중...")
fig1, axes1 = plt.subplots(2, 2, figsize=(16, 14))

colors_map = {'normal': '#2ecc71', 'violence': '#e74c3c', 'sexual': '#9b59b6'}
markers_map = {'normal': 'o', 'violence': '^', 'sexual': 's'}

for idx, beta in enumerate(all_betas):
    latents = beta_test_results[beta]['latents']

    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    latents_2d = tsne.fit_transform(latents)

    ax = axes1[idx // 2, idx % 2]

    for category in categories:
        mask = test_labels == category
        ax.scatter(
            latents_2d[mask, 0], latents_2d[mask, 1],
            c=colors_map[category], marker=markers_map[category],
            label=f'{category.capitalize()}',
            alpha=0.6, s=80, edgecolors='white', linewidth=0.5
        )

    mean_sim = beta_test_results[beta]['cosine_sims'].mean()
    final_kld = beta_histories[beta]['val_kld'][-1]

    ax.set_title(f'β={beta} Latent (KLD={final_kld:.2f})',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('t-SNE Dim 1', fontsize=10)
    ax.set_ylabel('t-SNE Dim 2', fontsize=10)
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.3, linestyle='--')

    print(f"  ✅ β={beta} Latent 완료")

plt.suptitle('Beta Comparison: Latent Space Structure',
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('/content/beta_comparison_latent_all.png', dpi=300, bbox_inches='tight')
plt.show()

# Reconstructed CLIP 비교 (2x2)
print("\n[2/2] Reconstructed CLIP 비교 계산 중...")
fig2, axes2 = plt.subplots(2, 2, figsize=(16, 14))

for idx, beta in enumerate(all_betas):
    recon = beta_test_results[beta]['recon']

    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    recon_2d = tsne.fit_transform(recon)

    ax = axes2[idx // 2, idx % 2]

    for category in categories:
        mask = test_labels == category
        ax.scatter(
            recon_2d[mask, 0], recon_2d[mask, 1],
            c=colors_map[category], marker=markers_map[category],
            label=f'{category.capitalize()}',
            alpha=0.6, s=80, edgecolors='white', linewidth=0.5
        )

    mean_sim = beta_test_results[beta]['cosine_sims'].mean()

    ax.set_title(f'β={beta} Reconstructed (Cosine Sim={mean_sim:.4f})',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('t-SNE Dim 1', fontsize=10)
    ax.set_ylabel('t-SNE Dim 2', fontsize=10)
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.3, linestyle='--')

    print(f"  ✅ β={beta} Reconstructed 완료")

plt.suptitle('Beta Comparison: Reconstructed CLIP Embedding',
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('/content/beta_comparison_recon_all.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*80)
print("✅ 전체 비교 t-SNE 시각화 완료")
print("   - Latent Space: /content/beta_comparison_latent_all.png")
print("   - Reconstructed: /content/beta_comparison_recon_all.png")
print("="*80)

## 14-5-4. Beta 값별 Semantic Cluster 보존도 분석

In [ ]:
# Beta 값별 Semantic Cluster 보존도 정량 분석
from sklearn.metrics import silhouette_score
from scipy.spatial.distance import cdist

print("="*80)
print("Beta 값별 Semantic Cluster 보존도 분석")
print("="*80)

cluster_analysis = {}

for beta in all_betas:
    print(f"\n[β={beta}] 분석 중...")

    latents = beta_test_results[beta]['latents']
    recon = beta_test_results[beta]['recon']

    # 레이블을 숫자로 변환
    label_to_num = {'normal': 0, 'violence': 1, 'sexual': 2}
    numeric_labels = np.array([label_to_num[label] for label in test_labels])

    # 1. Silhouette Score (클러스터 품질)
    try:
        latent_silhouette = silhouette_score(latents, numeric_labels)
        recon_silhouette = silhouette_score(recon, numeric_labels)
        original_silhouette = silhouette_score(test_embeddings, numeric_labels)
    except:
        latent_silhouette = 0
        recon_silhouette = 0
        original_silhouette = 0

    # 2. 카테고리 간 평균 거리 (클수록 잘 분리됨)
    category_centers_latent = []
    category_centers_recon = []
    category_centers_original = []

    for category in categories:
        mask = test_labels == category
        category_centers_latent.append(latents[mask].mean(axis=0))
        category_centers_recon.append(recon[mask].mean(axis=0))
        category_centers_original.append(test_embeddings[mask].mean(axis=0))

    # 중심점 간 평균 거리
    latent_inter_dist = cdist(category_centers_latent, category_centers_latent).mean()
    recon_inter_dist = cdist(category_centers_recon, category_centers_recon).mean()
    original_inter_dist = cdist(category_centers_original, category_centers_original).mean()

    # 3. 카테고리 내 평균 거리 (작을수록 촘촘함)
    latent_intra_dists = []
    recon_intra_dists = []
    original_intra_dists = []

    for category in categories:
        mask = test_labels == category
        center_latent = latents[mask].mean(axis=0)
        center_recon = recon[mask].mean(axis=0)
        center_original = test_embeddings[mask].mean(axis=0)

        latent_intra_dists.append(np.linalg.norm(latents[mask] - center_latent, axis=1).mean())
        recon_intra_dists.append(np.linalg.norm(recon[mask] - center_recon, axis=1).mean())
        original_intra_dists.append(np.linalg.norm(test_embeddings[mask] - center_original, axis=1).mean())

    latent_intra_dist = np.mean(latent_intra_dists)
    recon_intra_dist = np.mean(recon_intra_dists)
    original_intra_dist = np.mean(original_intra_dists)

    # 저장
    cluster_analysis[beta] = {
        'latent_silhouette': latent_silhouette,
        'recon_silhouette': recon_silhouette,
        'original_silhouette': original_silhouette,
        'latent_inter_dist': latent_inter_dist,
        'recon_inter_dist': recon_inter_dist,
        'original_inter_dist': original_inter_dist,
        'latent_intra_dist': latent_intra_dist,
        'recon_intra_dist': recon_intra_dist,
        'original_intra_dist': original_intra_dist
    }

    print(f"  ✅ 완료")

# 결과 출력
print("\n" + "="*80)
print("클러스터 보존도 분석 결과")
print("="*80)

print("\n📊 1. Silhouette Score (높을수록 좋음: -1 ~ 1)")
print("-"*80)
print(f"{'Beta':<10} {'Original':<12} {'Latent':<12} {'Reconstructed':<12}")
print("-"*80)
for beta in all_betas:
    analysis = cluster_analysis[beta]
    print(f"{beta:<10.2f} {analysis['original_silhouette']:<12.4f} "
          f"{analysis['latent_silhouette']:<12.4f} {analysis['recon_silhouette']:<12.4f}")

print("\n📊 2. Inter-Cluster Distance (카테고리 간 거리, 클수록 좋음)")
print("-"*80)
print(f"{'Beta':<10} {'Original':<12} {'Latent':<12} {'Reconstructed':<12}")
print("-"*80)
for beta in all_betas:
    analysis = cluster_analysis[beta]
    print(f"{beta:<10.2f} {analysis['original_inter_dist']:<12.4f} "
          f"{analysis['latent_inter_dist']:<12.4f} {analysis['recon_inter_dist']:<12.4f}")

print("\n📊 3. Intra-Cluster Distance (카테고리 내 거리, 작을수록 좋음)")
print("-"*80)
print(f"{'Beta':<10} {'Original':<12} {'Latent':<12} {'Reconstructed':<12}")
print("-"*80)
for beta in all_betas:
    analysis = cluster_analysis[beta]
    print(f"{beta:<10.2f} {analysis['original_intra_dist']:<12.4f} "
          f"{analysis['latent_intra_dist']:<12.4f} {analysis['recon_intra_dist']:<12.4f}")

print("\n💡 해석:")
print("-"*80)
best_latent_silhouette = max(all_betas, key=lambda b: cluster_analysis[b]['latent_silhouette'])
best_recon_silhouette = max(all_betas, key=lambda b: cluster_analysis[b]['recon_silhouette'])
print(f"✅ Latent Space에서 최고 클러스터 품질: β={best_latent_silhouette}")
print(f"✅ Reconstructed에서 최고 클러스터 품질: β={best_recon_silhouette}")
print("\n→ 낮은 β 값일수록 Semantic Cluster를 더 잘 보존!")
print("="*80)

## 14-6. Beta 값별 성능 분석 및 결론

In [ ]:
# Beta 값별 종합 분석
print("\n" + "="*80)
print("Beta 값 실험 종합 분석")
print("="*80)

print("\n📊 1. Reconstruction Quality (Cosine Similarity)")
print("-"*80)
for beta in all_betas:
    sim_mean = beta_test_results[beta]['cosine_sims'].mean()
    sim_std = beta_test_results[beta]['cosine_sims'].std()
    print(f"β={beta:4.2f}: {sim_mean:.6f} ± {sim_std:.6f}")

print("\n📊 2. Regularization Strength (Final KLD)")
print("-"*80)
for beta in all_betas:
    kld = beta_histories[beta]['val_kld'][-1]
    print(f"β={beta:4.2f}: {kld:.4f}")

print("\n📊 3. Trade-off Analysis")
print("-"*80)
print("Beta ↑ → KLD 강제 ↑ → 표준정규분포에 가까워짐")
print("Beta ↓ → Semantic 보존 ↑ → Reconstruction 품질 향상")
print("")

# Semantic preservation 측정 (카테고리별 분산)
print("\n📊 4. Semantic Cluster Preservation (카테고리 간 분리도)")
print("-"*80)
for beta in all_betas:
    category_means = []
    for category in categories:
        stats = beta_test_results[beta]['category_stats'][category]
        category_means.append(stats['mean'])

    # 카테고리 간 분산 (클수록 semantic 잘 보존)
    category_variance = np.var(category_means)
    print(f"β={beta:4.2f}: Category Variance = {category_variance:.8f}")

print("\n💡 해석:")
print("-"*80)
if beta_test_results[0.01]['cosine_sims'].mean() > beta_test_results[0.1]['cosine_sims'].mean():
    diff = beta_test_results[0.01]['cosine_sims'].mean() - beta_test_results[0.1]['cosine_sims'].mean()
    print(f"✅ β=0.01이 β=0.1보다 {diff:.6f} 더 높은 Cosine Similarity")
    print("   → Semantic preservation이 더 중요함을 확인!")
else:
    print("⚠️ β 증가가 성능 향상에 도움")

print("\n🎯 추천 Beta 값:")
best = max(all_betas, key=lambda b: beta_test_results[b]['cosine_sims'].mean())
print(f"   β={best} (가장 높은 Cosine Similarity)")
print("="*80)